In [1]:
# CMS_Gold_Aggregation
# Note: This notebook re-ingests from the CMS API and applies Silver 
# transformations inline due to Fabric trial cross-lakehouse file access 
# limitations. In a production environment, this notebook would read 
# directly from the Silver Delta tables via the shared Lakehouse connection.

import pandas as pd
import requests

# Re-pull from CMS API and re-transform to Silver
headers = {"Accept-Encoding": "identity"}
params = {"limit": 500, "offset": 0, "count": "true"}

response = requests.get(
    "https://openpaymentsdata.cms.gov/api/1/datastore/query/fb3a65aa-c901-4a38-a813-b04b00dfa2a9/0",
    params=params, headers=headers
)

df = pd.DataFrame(response.json()['results'])

# Apply Silver transformations
silver_cols = [
    'covered_recipient_type', 'covered_recipient_first_name',
    'covered_recipient_last_name', 'covered_recipient_npi',
    'covered_recipient_specialty_1', 'recipient_city', 'recipient_state',
    'submitting_applicable_manufacturer_or_applicable_gpo_name',
    'total_amount_of_payment_usdollars', 'date_of_payment',
    'form_of_payment_or_transfer_of_value',
    'nature_of_payment_or_transfer_of_value', 'program_year'
]

df_silver = df[silver_cols].copy()
df_silver.columns = [
    'recipient_type', 'first_name', 'last_name', 'npi', 'specialty',
    'city', 'state', 'manufacturer_name', 'payment_amount',
    'payment_date', 'payment_form', 'payment_nature', 'program_year'
]

df_silver['payment_amount'] = pd.to_numeric(df_silver['payment_amount'], errors='coerce')
df_silver['payment_date'] = pd.to_datetime(df_silver['payment_date'], errors='coerce')

# GOLD TABLE 1 - Payments by Manufacturer
gold_by_manufacturer = df_silver.groupby('manufacturer_name').agg(
    total_payments=('payment_amount', 'sum'),
    avg_payment=('payment_amount', 'mean'),
    payment_count=('payment_amount', 'count')
).reset_index().sort_values('total_payments', ascending=False)

# GOLD TABLE 2 - Payments by State
gold_by_state = df_silver.groupby('state').agg(
    total_payments=('payment_amount', 'sum'),
    avg_payment=('payment_amount', 'mean'),
    payment_count=('payment_amount', 'count')
).reset_index().sort_values('total_payments', ascending=False)

# GOLD TABLE 3 - Payments by Specialty
gold_by_specialty = df_silver.groupby('specialty').agg(
    total_payments=('payment_amount', 'sum'),
    avg_payment=('payment_amount', 'mean'),
    payment_count=('payment_amount', 'count')
).reset_index().sort_values('total_payments', ascending=False)

print(f"Silver records: {len(df_silver)}")
print("\nTop 5 Manufacturers:")
display(gold_by_manufacturer.head())
print("\nTop 5 States:")
display(gold_by_state.head())
print("\nTop 5 Specialties:")
display(gold_by_specialty.head())

# Write Gold tables to Lakehouse
gold_by_manufacturer.to_parquet("/lakehouse/default/Files/gold_by_manufacturer.parquet", index=False)
gold_by_state.to_parquet("/lakehouse/default/Files/gold_by_state.parquet", index=False)
gold_by_specialty.to_parquet("/lakehouse/default/Files/gold_by_specialty.parquet", index=False)

print("\nGold tables written successfully:")
print(f"  gold_by_manufacturer: {len(gold_by_manufacturer)} rows")
print(f"  gold_by_state: {len(gold_by_state)} rows")
print(f"  gold_by_specialty: {len(gold_by_specialty)} rows")

Silver records: 500

Top 5 Manufacturers:



Top 5 States:



Top 5 Specialties:



Gold tables written successfully:
  gold_by_manufacturer: 2 rows
  gold_by_state: 39 rows
  gold_by_specialty: 36 rows


In [2]:
import os

# Check what lakehouses are mounted
print(os.listdir('/lakehouse'))
print(os.listdir('/lakehouse/default/Files'))

['default']
['gold_by_manufacturer.parquet', 'gold_by_specialty.parquet', 'gold_by_state.parquet']


In [3]:
import os

for root, dirs, files in os.walk('/lakehouse'):
    for f in files:
        print(os.path.join(root, f))
    for d in dirs:
        print(os.path.join(root, d))

/lakehouse/default
/lakehouse/default/Tables
/lakehouse/default/Files
/lakehouse/default/Files/gold_by_manufacturer.parquet
/lakehouse/default/Files/gold_by_specialty.parquet
/lakehouse/default/Files/gold_by_state.parquet


In [4]:
import pandas as pd
import requests

# Re-pull from CMS API and re-transform to Silver
# (since we can't access the other lakehouse's files directly)

headers = {"Accept-Encoding": "identity"}
params = {"limit": 500, "offset": 0, "count": "true"}

response = requests.get(
    "https://openpaymentsdata.cms.gov/api/1/datastore/query/fb3a65aa-c901-4a38-a813-b04b00dfa2a9/0",
    params=params, headers=headers
)

df = pd.DataFrame(response.json()['results'])

# Apply Silver transformations
silver_cols = [
    'covered_recipient_type', 'covered_recipient_first_name',
    'covered_recipient_last_name', 'covered_recipient_npi',
    'covered_recipient_specialty_1', 'recipient_city', 'recipient_state',
    'submitting_applicable_manufacturer_or_applicable_gpo_name',
    'total_amount_of_payment_usdollars', 'date_of_payment',
    'form_of_payment_or_transfer_of_value',
    'nature_of_payment_or_transfer_of_value', 'program_year'
]

df_silver = df[silver_cols].copy()
df_silver.columns = [
    'recipient_type', 'first_name', 'last_name', 'npi', 'specialty',
    'city', 'state', 'manufacturer_name', 'payment_amount',
    'payment_date', 'payment_form', 'payment_nature', 'program_year'
]

df_silver['payment_amount'] = pd.to_numeric(df_silver['payment_amount'], errors='coerce')
df_silver['payment_date'] = pd.to_datetime(df_silver['payment_date'], errors='coerce')

# GOLD TABLE 1 - Payments by Manufacturer
gold_by_manufacturer = df_silver.groupby('manufacturer_name').agg(
    total_payments=('payment_amount', 'sum'),
    avg_payment=('payment_amount', 'mean'),
    payment_count=('payment_amount', 'count')
).reset_index().sort_values('total_payments', ascending=False)

# GOLD TABLE 2 - Payments by State
gold_by_state = df_silver.groupby('state').agg(
    total_payments=('payment_amount', 'sum'),
    avg_payment=('payment_amount', 'mean'),
    payment_count=('payment_amount', 'count')
).reset_index().sort_values('total_payments', ascending=False)

# GOLD TABLE 3 - Payments by Specialty
gold_by_specialty = df_silver.groupby('specialty').agg(
    total_payments=('payment_amount', 'sum'),
    avg_payment=('payment_amount', 'mean'),
    payment_count=('payment_amount', 'count')
).reset_index().sort_values('total_payments', ascending=False)

print(f"Silver records: {len(df_silver)}")
print("\nTop 5 Manufacturers:")
display(gold_by_manufacturer.head())
print("\nTop 5 States:")
display(gold_by_state.head())
print("\nTop 5 Specialties:")
display(gold_by_specialty.head())

Silver records: 500

Top 5 Manufacturers:



Top 5 States:



Top 5 Specialties:


In [5]:
# Write Gold tables to Lakehouse
gold_by_manufacturer.to_parquet("/lakehouse/default/Files/gold_by_manufacturer.parquet", index=False)
gold_by_state.to_parquet("/lakehouse/default/Files/gold_by_state.parquet", index=False)
gold_by_specialty.to_parquet("/lakehouse/default/Files/gold_by_specialty.parquet", index=False)

print("Gold tables written successfully:")
print(f"  gold_by_manufacturer: {len(gold_by_manufacturer)} rows")
print(f"  gold_by_state: {len(gold_by_state)} rows")
print(f"  gold_by_specialty: {len(gold_by_specialty)} rows")

Gold tables written successfully:
  gold_by_manufacturer: 2 rows
  gold_by_state: 39 rows
  gold_by_specialty: 36 rows


In [6]:
# Write Gold tables to Bronze Lakehouse using explicit path
import os

# Check what path is available
print(os.listdir('/lakehouse/default/Files'))

# Write Gold files
gold_by_manufacturer.to_parquet("/lakehouse/default/Files/gold_by_manufacturer.parquet", index=False)
gold_by_state.to_parquet("/lakehouse/default/Files/gold_by_state.parquet", index=False)
gold_by_specialty.to_parquet("/lakehouse/default/Files/gold_by_specialty.parquet", index=False)

print("Gold files written:")
print(os.listdir('/lakehouse/default/Files'))

['gold_by_manufacturer.parquet', 'gold_by_specialty.parquet', 'gold_by_state.parquet']
Gold files written:
['gold_by_manufacturer.parquet', 'gold_by_specialty.parquet', 'gold_by_state.parquet']


In [7]:
import os

# Check where we actually are
print("Default Files:")
print(os.listdir('/lakehouse/default/Files'))

# Check the full lakehouse path
for root, dirs, files in os.walk('/lakehouse'):
    for f in files:
        if 'gold' in f.lower():
            print(f"Found: {os.path.join(root, f)}")

Default Files:
['gold_by_manufacturer.parquet', 'gold_by_specialty.parquet', 'gold_by_state.parquet']
Found: /lakehouse/default/Files/gold_by_manufacturer.parquet
Found: /lakehouse/default/Files/gold_by_specialty.parquet
Found: /lakehouse/default/Files/gold_by_state.parquet
